In [21]:
%pip install dotenv

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\SAGAR\.pyenv\pyenv-win\versions\3.8.10\python.exe -m pip install --upgrade pip' command.


In [22]:
import os
import json
import re
from typing import List, Dict, Any, Tuple
from pathlib import Path
from datetime import datetime

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from groq import Groq
from transformers import pipeline
from dotenv import load_dotenv

load_dotenv()

# ─────────────────────────────────────────────
# CONFIG & MODELS
# ─────────────────────────────────────────────

app = FastAPI(title="Indy ADHD Copilot API")

# Initialize Models once on startup
print("Loading NLP model...")
nlp_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

GROQ_API_KEY = os.getenv('GROQ_API_KEY')
client = Groq(api_key=GROQ_API_KEY)
LLM_MODEL = "groq/compound-mini"  # Fixed: was "groq/compound-mini" which doesn't exist
SESSIONS_DIR = Path("sessions")
SESSIONS_DIR.mkdir(exist_ok=True)

LABEL_DESCRIPTIONS = {
    "affective": "expressing emotions, feelings, distress, sadness, fear, love, anger",
    "cognitive": "thinking, reasoning, reflecting, understanding, analyzing, wondering",
    "agency":    "taking action, making decisions, expressing control, planning, choosing"
}

# ─────────────────────────────────────────────
# SCHEMAS
# ─────────────────────────────────────────────

class ChatRequest(BaseModel):
    session_id: int
    message: str
    history: List[Dict[str, str]]  # List of {"role": "user/assistant", "content": "..."}

# ─────────────────────────────────────────────
# LOGIC HELPERS
# ─────────────────────────────────────────────

def score_text(text: str):
    candidate_labels = list(LABEL_DESCRIPTIONS.values())
    result = nlp_classifier(text, candidate_labels, multi_label=False)
    
    scores = {}
    for label, score in zip(result['labels'], result['scores']):
        for clean_name, desc in LABEL_DESCRIPTIONS.items():
            if label == desc:
                scores[clean_name] = round(score, 4)
    
    dominant = max(scores, key=scores.get)
    return {"scores": scores, "dominant": dominant}

def get_past_context(session_id: int):
    # Same logic as your script: look for session_001.json etc.
    context_blocks = []
    for i in range(session_id - 1, max(0, session_id - 4), -1):
        path = SESSIONS_DIR / f"session_{i:03d}.json"
        if path.exists():
            with open(path, "r") as f:
                s = json.load(f)
                block = f"Session {s['session_id']}: Mood: {s.get('mood')}, Wins: {s.get('small_wins')}"
                context_blocks.append(block)
    return "\n".join(context_blocks) if context_blocks else "No history."

def load_or_create_session(session_id: int) -> dict:
    """Load session or create new one if doesn't exist"""
    path = SESSIONS_DIR / f"session_{session_id:03d}.json"
    if path.exists():
        with open(path, "r") as f:
            return json.load(f)
    else:
        return {
            "session_id": session_id,
            "created": datetime.now().isoformat(),
            "conversation": [],
            "cognitive_shifts": []
        }

def save_session(session_id: int, session_data: dict) -> None:
    """Save session to file"""
    path = SESSIONS_DIR / f"session_{session_id:03d}.json"
    with open(path, "w") as f:
        json.dump(session_data, f, indent=2)

def extract_notes_from_response(raw_reply: str) -> Tuple[str, Dict[str, Any]]:
    """
    Extract the text response and JSON notes from the LLM reply.
    Handles multiple formats and edge cases.
    Returns: (clean_text, notes_dict)
    """
    notes: Dict[str, Any] = {}
    clean_text = raw_reply
    
    # Try to find ### UPDATED_NOTES marker
    if "### UPDATED_NOTES" in raw_reply:
        clean_text = raw_reply.split("### UPDATED_NOTES")[0].strip()
        json_part = raw_reply.split("### UPDATED_NOTES")[-1].strip()
        
        # Try to extract JSON
        try:
            # Remove markdown code blocks if present
            json_part = re.sub(r"^```(?:json)?", "", json_part).strip()
            json_part = re.sub(r"```$", "", json_part).strip()
            
            # Try to find JSON object in the string (handles cases where there's extra text)
            json_match = re.search(r'\{.*\}', json_part, re.DOTALL)
            if json_match:
                json_str = json_match.group(0)
                notes = json.loads(json_str)
            else:
                # Try parsing the whole thing
                notes = json.loads(json_part)
        except json.JSONDecodeError as e:
            print(f"Warning: Failed to parse JSON notes: {e}")
            notes = {"error": "Failed to parse notes", "raw": json_part[:200]}
    else:
        # No marker found - try to find JSON object at the end anyway
        try:
            json_match = re.search(r'\{[^{}]*"[^"]*"[^{}]*\}', raw_reply, re.DOTALL)
            if json_match:
                json_str = json_match.group(0)
                parsed = json.loads(json_str)
                # If we found valid JSON at the end, assume it's notes
                clean_text = raw_reply[:json_match.start()].strip()
                notes = parsed
        except:
            pass
    
    return clean_text, notes

# ─────────────────────────────────────────────
# ENDPOINTS
# ─────────────────────────────────────────────

@app.post("/chat")
async def chat_endpoint(req: ChatRequest):
    try:
        # 1. Cognitive Shift Analysis (The 'NLP' Layer)
        analysis = score_text(req.message)
        
        # 2. Get RAG Context
        past_context = get_past_context(req.session_id)
        
        # 3. Construct System Prompt - VERY explicit instructions
                # 3. Construct System Prompt
        system_prompt = f"""You are Indy, an ADHD copilot. A kind, empathetic listener who helps people with ADHD.
Be gentle, short, NO GUILT. Respond in the same language as the user.

EXISTING NOTES (everything you know about this person so far):
{past_context}

YOUR JOB:
1. Respond warmly and with empathy (SHORT - max 2 sentences)
2. Listen deeply and decide what is worth remembering about this person
3. Output updated notes that MERGE your new observations with the existing ones

WHAT TO CAPTURE — use your judgment. You are building a living psychological and practical profile of this person.
Think like a thoughtful therapist and personal assistant combined. You might capture things like:
- Dreams, aspirations, things they want for their life
- Sources of trauma, past wounds, painful memories
- Triggers — what makes their ADHD or anxiety spike
- Coping patterns — what they do (helpful or not) when overwhelmed
- Relationships — people who matter, tensions, support systems
- Core beliefs about themselves ("I'm lazy", "I always fail")
- Current blockers, fears, avoidances
- Wins, moments of pride, growth
- Recurring themes across conversations
- Anything else that feels important to remember about this person

You are NOT limited to a fixed set of categories. Invent whatever keys and structure best captures what you learned.

HOW TO OUTPUT NOTES:
- Take the EXISTING NOTES above as your base
- Add new observations, update or enrich existing ones
- Remove nothing unless it was clearly corrected by the user
- Each note entry should have: title, content, category (you decide the category name)
- Use snake_case for all keys
- session_id, date, and session_type must always be at the root

OUTPUT FORMAT:
Write your short empathetic reply first.
Then on a new line write:
### UPDATED_NOTES followed by the complete merged JSON object.

RULES:
- Valid JSON only — no markdown code fences, no trailing commas
- The JSON must be COMPLETE (include all old notes + new ones)
- If nothing new to add, output the existing notes unchanged
- session_id = {req.session_id}
- date = "{datetime.now().strftime('%Y-%m-%d')}"
- session_type = "brain_dump"
"""

        # 4. Call Groq with full conversation history for context
        messages = [{"role": "system", "content": system_prompt}] + req.history + [{"role": "user", "content": req.message}]
        
        try:
            completion = client.chat.completions.create(
                model=LLM_MODEL,
                messages=messages,
                temperature=0.7,
                max_tokens=1024
            )
            
            raw_reply = completion.choices[0].message.content
        except Exception as groq_error:
            print(f"Groq API Error: {groq_error}")
            raise HTTPException(status_code=502, detail=f"Groq API error: {str(groq_error)}")
        
        # 5. Parsing & Cleaning - now more robust
        clean_text, notes = extract_notes_from_response(raw_reply)
        
        # Ensure we always have at least empty notes
        if not notes:
            notes = {
                "session_id": req.session_id,
                "date": datetime.now().strftime("%Y-%m-%d"),
                "session_type": "brain_dump",
                "priorities": [],
                "blockers": [],
                "small_wins": [],
                "insights": []
            }

        # 6. Save session with conversation history
        session_data = load_or_create_session(req.session_id)
        
        # Append user message to conversation
        session_data["conversation"].append({
            "role": "user",
            "content": req.message,
            "timestamp": datetime.now().isoformat()
        })
        
        # Append assistant response to conversation
        session_data["conversation"].append({
            "role": "assistant",
            "content": clean_text,
            "timestamp": datetime.now().isoformat(),
            "cognitive_shift": analysis,
            "notes": notes
        })
        
        # Track cognitive shifts for later analysis
        session_data["cognitive_shifts"].append({
            "message": req.message,
            "analysis": analysis,
            "timestamp": datetime.now().isoformat()
        })
        
        save_session(req.session_id, session_data)
        
        return {
            "reply": clean_text,
            "cognitive_shift": analysis,
            "extracted_notes": notes
        }

    except HTTPException:
        raise
    except Exception as e:
        print(f"Error in chat endpoint: {e}")
        import traceback
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))

Loading NLP model...


In [23]:
import json

def merge_notes(existing_notes: dict, llm_response: str) -> dict:
    """
    Parses the UPDATED_NOTES JSON from the LLM response and merges it
    with the existing notes. The LLM is expected to return the full
    merged object, so this function simply validates and returns it.
    Falls back to existing notes if parsing fails.
    """
    marker = "### UPDATED_NOTES"
    if marker not in llm_response:
        return existing_notes

    raw_json = llm_response.split(marker, 1)[1].strip()

    try:
        updated = json.loads(raw_json)
        # Ensure the new notes dict is merged on top of existing notes
        # so no old keys are silently dropped if the LLM forgets them
        if "notes" in existing_notes and "notes" in updated:
            merged_notes = {**existing_notes.get("notes", {}), **updated["notes"]}
            updated["notes"] = merged_notes
        return updated
    except json.JSONDecodeError:
        # If the LLM returned malformed JSON, preserve what we had
        return existing_notes

In [ ]:
# Ensure nest_asyncio is available in the notebook environment so uvicorn can run
# without raising "asyncio.run() cannot be called from a running event loop".
# The %pip magic is appropriate inside a Jupyter cell.
%pip install -q nest_asyncio

import nest_asyncio
nest_asyncio.apply()

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)

You should consider upgrading via the 'c:\Users\SAGAR\.pyenv\pyenv-win\versions\3.8.10\python.exe -m pip install --upgrade pip' command.


Note: you may need to restart the kernel to use updated packages.


INFO:     Started server process [12516]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:54598 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:54598 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:54598 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:54598 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:56708 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:56708 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:56708 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:56708 - "POST /chat HTTP/1.1" 200 OK
